In [1]:
import sys
import os

# Adds 'src/capstone' to the path so 'pipeline' is discoverable
# assuming the notebook is in ROOT/notebooks/ and code is in ROOT/src/capstone/
sys.path.append(os.path.abspath("../"))

import pandas as pd

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory

## Config

`use_synthetic=True, target_real_pct=5/6` → SyntheticAugmenter appends ~20% synthetic
rows to `X_train` (formula: `floor(real / target_real_pct) - real`).
No snapshot flags set → `build()` leaves all version counters unchanged.
This run will read from GCS but write nothing back.

In [2]:
config = (
    VersionConfig
    .load(use_synthetic=False)
    .use_baselines_version(version="3.2")
    .snapshot_final()
    .build()
)
run = PipelineRun(config)
stages = PipelineFactory.retrain_existing_data(config)

print('\nScenario:', stages.scenario)
print('raw_version   :', config.raw_version)
print('model_version :', config.model_version)

No versions.json found in GCS — using defaults. Call config.commit() after your first snapshot to persist versions.
VersionConfig loaded:
  data:          v3.1 (raw_suffix='real')
  baselines:     v3.1
  model:         v3.1
  hyperparams:   v1.0
  use_synthetic: False

VersionConfig ready:
  Active flags      : ['final']
  data              : v3.1 -> v3.2
  baselines         : v3.0 (unchanged)  [pinned -> v3.2]
  model             : v3.1 (unchanged)
  hyperparams       : v1.0 (unchanged)
  raw_version       : v3.2_real
  final_version     : v3.2_100real
  model_version     : v3.1
  baselines_version : v3.2
  hyperparam_version: v1.0
  use_synthetic     : False

  Call config.commit() after all snapshots succeed.

Scenario: retrain_existing_data
raw_version   : v3.2_real
model_version : v3.1


## First-time Setup: Create the Locked Validation Holdout

**Run once**, before any pipeline scenario. Skip if `splits/validation_ids.json` already
exists in GCS — `DataSplitter` detects the file and errors helpfully if it's missing.

```bash
python scripts/create_validation_set.py        # dry-run (default)
python scripts/create_validation_set.py --yes  # write to GCS
```

The script runs Load → DataPreprocessor → FeatureEngineer, then creates a stratified
holdout using the 18-cell key (vertical × tier × above_baseline).

---
## Main Flow

Runs the full `retrain_existing_data` scenario with synthetic augmentation.

## Stage 1 — DataLoader (GCS)

Reads the parquet snapshot at `config.raw_version`. No BigQuery call.

In [3]:
stages.loader.run(run)
run.summary()

Loaded snapshot 'v3.2_real': 58037 rows from 2026-04-27
  Polls: {'upload': 20468, '24h': 20048, '7d': 17521}
Loaded baselines 'v3.2': 28814 baseline videos, 974 median rows (974 channels)
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), baselines=(3, 0), hyperparams=(1, 0), use_synthetic=False, active=['final']))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  y_train        None
  y_test         None
  y_val          None
  models         empty dict
  results        empty dict


## Stage 2 — DataPreprocessor

Pivots the long-format poll records into one row per video (via `pivot_snapshots`),
joins channel baselines, and runs structural cleanup via `build_clean_dataset`.
Writes `run.df_clean`.

In [4]:
stages.preprocessor.run(run)
run.summary()

Building clean dataset
snapshot cols: Index(['video_id', 'poll_timestamp', 'channel_id', 'channel_handle', 'title',
       'view_count', 'like_count', 'comment_count', 'face_count', 'brightness',
       'colorfulness', 'vertical', 'tier', 'description', 'tags',
       'duration_seconds', 'category_id', 'category_name', 'published_at',
       'poll_label', 'hours_since_publish', 'subscriber_count',
       'contains_synthetic_media'],
      dtype='object')

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 17463 (dropped 3052 incomplete)
  Pivoted shape: (17532, 34)

[2/3] Joining baseline medians...
  Baseline join: 17532/17532 videos matched a channel median

[3/3] Cleaning data...
  Cleaned: 17532 rows × 40 columns

Clean dataset: 17532 rows × 40 columns
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), baselines=(3, 0), hyperparams=(1, 0), use_synthetic=False, active=['final']))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFra

## Stage 3 — FeatureEngineer

Runs the full feature engineering chain on `run.df_clean` (one row per video).
Computes `above_baseline` target, all ratio/growth features, and categorical encodings.
Writes `run.df_engineered`.

In [5]:
stages.engineer.run(run)
run.summary()

  all: dropped 396 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/9] Computing target variable...
  Target distribution: 55.1% above baseline, 44.9% below

[2/9] Computing velocity features...
  Computed velocity, upload momentum, normalized velocity, and acceleration features

[3/9] Computing ratio and baseline-normalized features...
  Computed ratio and baseline-normalized features

[4/9] Computing subscriber-normalized metrics...
  Computed subscriber-normalized metrics for upload/24h/7d

[5/9] Computing categorical features...
  Title categories:
title_category
neutral        10835
all_caps        1931
exclamation     1861
question        1498
listicle         527
how_to           219
clickbait        193
emoji_heavy       72
  Description categories:
desc_category
link_heavy        4519
minimal           4297
has_links         3656
has_hashtags      2559
neutral           1323
has_timestamps     736
long_form           4

## Stage 4 — DataSplitter

Loads the locked validation IDs from GCS (`splits/validation_ids.json`).
Filters `run.df_engineered` to produce `df_val`, then stratifies the remaining
rows into `df_train` / `df_test` (80/20). Derives unscaled `X_train`, `X_test`,
`X_val` and corresponding `y_` series.

In [6]:
stages.splitter.run(run)
run.summary()

DataSplitter — loaded holdout (5,193 val rows):
  df_val:    5,193 rows (30.3%)
  df_train:  9,554 rows (55.8%)
  df_test:   2,389 rows (13.9%)
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), baselines=(3, 0), hyperparams=(1, 0), use_synthetic=False, active=['final']))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(9554, 53)
  X_test         populated  DataFrame shape=(2389, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  None
  y_train        populated  Series length=9554
  y_test         populated  Series length

## Stage 5 — Scaler

Fits `StandardScaler` on `X_train`, transforms `X_train`, `X_test`, and `X_val`.
Captures `run.X_val_unscaled` so Validator can apply each loaded model's own
historical scaler for a fair apples-to-apples comparison.

In [7]:
stages.scaler.run(run)
run.summary()

PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), baselines=(3, 0), hyperparams=(1, 0), use_synthetic=False, active=['final']))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(9554, 53)
  X_test         populated  DataFrame shape=(2389, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  populated  DataFrame shape=(5193, 53)
  y_train        populated  Series length=9554
  y_test         populated  Series length=2389
  y_val          populated  Series length=5193
  models         empty dict
  results        empty dict


## Stage 6 — SyntheticAugmenter

Generates synthetic rows from `run.df_train` via `generate_synthetic_data`,
engineers them through the same `FeatureEngineerLogic` chain, aligns with
`X_train`'s column space, scales with the fitted `StandardScaler`, then
concatenates onto `run.X_train` / `run.y_train`. `X_test` and `X_val` are
never touched. Target: ~20% of real train rows added as synthetic
(`target_real_pct=5/6` → `floor(real/0.8333) - real ≈ 0.2 * real`).

In [8]:
stages.augmenter.run(run)
run.summary()

[SyntheticAugmenter] use_synthetic=False — skipping.
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), baselines=(3, 0), hyperparams=(1, 0), use_synthetic=False, active=['final']))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(9554, 53)
  X_test         populated  DataFrame shape=(2389, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  populated  DataFrame shape=(5193, 53)
  y_train        populated  Series length=9554
  y_test         populated  Series length=2389
  y_val          populated  Series length=5193
  mod

## Stage 7 — FinalSnapshotter

Persists the six per-split modeling artifacts (`X_train`, `y_train`, `X_test`,
`y_test`, `X_val`, `y_val`) to GCS at `config.final_version`. Sits after
SyntheticAugmenter so `X_train` / `y_train` include synthetic rows when
`use_synthetic=True`. No-op when `config.take_snapshot_final` is False.

In [9]:
stages.final_snapshotter.run(run)

  Uploaded gs://maduros-dolce-capstone-data/snapshots/splits_v3.2_100real_X_train_9554rows_20260428_054153.parquet
  Uploaded gs://maduros-dolce-capstone-data/snapshots/splits_v3.2_100real_y_train_9554rows_20260428_054153.parquet
  Uploaded gs://maduros-dolce-capstone-data/snapshots/splits_v3.2_100real_X_test_2389rows_20260428_054153.parquet
  Uploaded gs://maduros-dolce-capstone-data/snapshots/splits_v3.2_100real_y_test_2389rows_20260428_054153.parquet
  Uploaded gs://maduros-dolce-capstone-data/snapshots/splits_v3.2_100real_X_val_5193rows_20260428_054153.parquet
  Uploaded gs://maduros-dolce-capstone-data/snapshots/splits_v3.2_100real_y_val_5193rows_20260428_054153.parquet

--- Splits Snapshot v3.2_100real ---
  X_train: 9554 rows
  y_train: 9554 rows
  X_test: 2389 rows
  y_test: 2389 rows
  X_val: 5193 rows
  y_val: 5193 rows
[FinalSnapshotter] Splits snapshot 'v3.2_100real' saved to GCS.


PipelineRun(model_version='v3.1', num_synth_rows=0, populated=['df_videos', 'df_baselines', 'df_medians', 'df_clean', 'df_engineered', 'df_train', 'df_test', 'df_val', 'X_train', 'X_test', 'X_val', 'X_val_unscaled', 'y_train', 'y_test', 'y_val'])

## Stage 8 — ModelLoader

Auto-discovers all `models/<model_version>_*/` directories in GCS and loads each
saved model, scaler, and feature column list into `run.models`.

In [10]:
stages.model_loader.run(run)
print('\nLoaded models:', list(run.models.keys()))

Loading 3 models for base version 'v3.1': ['v3.1_lr_l1', 'v3.1_rf', 'v3.1_xgb']
Loaded model 'v3.1_lr_l1' (LogisticRegression)
  ROC-AUC: 0.7715
Loaded model 'v3.1_rf' (RandomForest)
  ROC-AUC: 0.8517
Loaded model 'v3.1_xgb' (XGBoost)
  ROC-AUC: 0.884

Loaded models: ['lr_l1', 'rf', 'xgb']


## Stage 9 — Validator

Evaluates each loaded model on the locked validation set (`X_val_unscaled` +
each model's own historical scaler). Populates `run.results` with AUC, accuracy,
F1, precision, recall, and top feature importances per model.

In [11]:
stages.validator.run(run)
run.summary()

[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']
[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']


/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']

=== Validator — validation-set results ===
  lr_l1           AUC=0.7476  acc=0.6905  F1↑=0.7321
  rf              AUC=0.8698  acc=0.7953  F1↑=0.8251
  xgb             AUC=0.9070  acc=0.8300  F1↑=0.8551
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), baselines=(3, 0), hyperparams=(1, 0), use_synthetic=False, active=['final']))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(9554, 53)
  X_test         populated  DataFrame shape=(2389, 53)
  X_val          pop

## Stage 10 — Persist Validation Results to GCS

In [12]:
stages.validation_results_snapshotter.run(run)
# Appends to gs://maduros-dolce-capstone-data/models/{model_version}/validation_results.jsonl

Validation results appended → gs://maduros-dolce-capstone-data/models/v3.1/validation_results.jsonl
  lr_l1           AUC=0.7476  acc=0.6905  F1↑=0.7321
  rf              AUC=0.8698  acc=0.7953  F1↑=0.8251
  xgb             AUC=0.9070  acc=0.8300  F1↑=0.8551


PipelineRun(model_version='v3.1', num_synth_rows=0, populated=['df_videos', 'df_baselines', 'df_medians', 'df_clean', 'df_engineered', 'df_train', 'df_test', 'df_val', 'X_train', 'X_test', 'X_val', 'X_val_unscaled', 'y_train', 'y_test', 'y_val', 'models', 'results'])

## Results

This is the **first validation-set evaluation** for this project — there is no prior
validation baseline in `data_analysis.ipynb` (only test-set scores were recorded there).

Cross-check model rank order and approximate AUC range against the test scores
from `data_analysis.ipynb`. Validation AUC will typically be slightly lower than
test AUC since the holdout was never touched during training.

In [13]:
metric_cols = [
    'roc_auc', 'accuracy',
    'f1_above', 'precision_above', 'recall_above',
    'f1_below', 'precision_below', 'recall_below',
]

df_results = (
    pd.DataFrame(run.results).T
    [metric_cols]
    .astype(float)
    .round(4)
)
df_results.index.name = 'model'

df_results.style.highlight_max(color='#d4edda').format('{:.4f}')

,roc_auc,accuracy,f1_above,precision_above,recall_above,f1_below,precision_below,recall_below
model,,,,,,,,
lr_l1,0.7476,0.6905,0.7321,0.7050,0.7614,0.6337,0.6689,0.6020
rf,0.8698,0.7953,0.8251,0.7850,0.8696,0.7532,0.8118,0.7025
xgb,0.9070,0.8300,0.8551,0.8116,0.9036,0.7942,0.8597,0.7380


In [14]:
# Top features for each model — sanity check against data_analysis.ipynb
for name, entry in run.results.items():
    top = entry.get('top_features', [])[:5]
    print(f'\n{name}:')
    for f in top:
        key = 'coefficient' if 'coefficient' in f else 'importance'
        print(f'  {f["feature"]:35s}  {f[key]:+.4f}')


lr_l1:
  like_rate_24h                        +3.8403
  view_count_upload_vs_baseline        -2.1137
  view_count_24h                       -1.7407
  like_count_24h                       +1.1326
  view_velocity_upload                 +0.8387

rf:
  like_rate_24h                        +0.0785
  view_count_24h                       +0.0494
  view_count_velocity_24h              +0.0438
  views_per_sub_24h                    +0.0416
  likes_per_sub_24h                    +0.0402

xgb:
  baseline_baseline_video_count        +0.0694
  like_rate_24h                        +0.0462
  like_count_24h                       +0.0353
  view_count_24h                       +0.0329
  is_short                             +0.0321


# 3-Way (Train, Test, Validation) Evaluation

`hist_test` = test metrics from original v3.1 training.
`curr_test` = same loaded models on today's test split.
`val` = locked holdout. 

TODO: To store to GCS long-term, extend ValidationResultsSnapshotter to also snapshot test_results_current

In [15]:
# Current test-set evaluation of loaded models (requires reconstructing X_test_unscaled)
import pandas as pd
from utils.snapshot_model import ModelResult
from dataclasses import asdict

X_test_unscaled = pd.DataFrame(
    stages.scaler.scaler_.inverse_transform(run.X_test),
    columns=run.X_test.columns,
    index=run.X_test.index,
)

test_results_current = {}
for name, entry in run.models.items():
    X = stages.validator.logic._prepare_X(entry, X_test_unscaled)
    result = ModelResult.from_sklearn(entry["model"], X, run.y_test, X.columns.tolist())
    test_results_current[name] = asdict(result)

metric_cols = ['roc_auc', 'accuracy', 'f1_above', 'precision_above', 'recall_above',
               'f1_below', 'precision_below', 'recall_below']

records = {}
for name in run.results:
    records[(name, 'hist_test')] = {m: run.models[name]["metadata"]["result"].get(m) for m in metric_cols}
    records[(name, 'curr_test')] = {m: test_results_current[name].get(m) for m in metric_cols}
    records[(name, 'val')]       = {m: run.results[name].get(m) for m in metric_cols}

df_3way = (
    pd.DataFrame(records).T
    .astype(float)
    .round(4)
)
df_3way.index.names = ['model', 'split']
df_3way.style.highlight_max(color='#d4edda').format('{:.4f}')

[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']
[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']


/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']


# Final: GCS Write 

Commit config changes to GCS.

In [16]:
config.commit()

Committed versions.json -> data v3.2, model v3.1, baselines v3.0, hyperparams v1.0
